In [ ]:
import pandas as pd
import numpy as np

In [ ]:
portfolio_df = pd.read_json('data/portfolio.json', orient='records', lines=True)
profile_df = pd.read_json('data/profile.json', orient='records', lines=True)
transcript_df = pd.read_json('data/transcript.json', orient='records', lines=True)

In [ ]:
portfolio_df

In [ ]:
# Rename 'id' column to 'offer_id' for clarity
portfolio_df = portfolio_df.rename(columns={'id': 'offer_id'})

# Convert 'channels' column to 4 separate binary columns (email, mobile, social, web)
for channel in ['email', 'mobile', 'social', 'web']:
    portfolio_df[f'{channel}'] = portfolio_df['channels'].apply(lambda x: 1 if channel in x else 0)

portfolio_df = portfolio_df.drop(columns=['channels'])


In [ ]:
profile_df

In [ ]:
profile_df = profile_df.rename(columns={'id': 'customer_id'}) # Rename 'id' column to 'customer_id'
profile_df['became_member_on'] = pd.to_datetime(profile_df['became_member_on'], format='%Y%m%d') # Convert 'became_member_on' to datetime format
profile_df.loc[profile_df['age'] == 118, 'age'] = np.nan # Replace all values of 118 in the 'age' column with NaN
profile_df = profile_df[['customer_id', 'gender', 'age', 'became_member_on', 'income']]

# Extract rows with any NaN values into a separate dataframe
anonymous_profile_df = profile_df[profile_df.isna().any(axis=1)].copy()
anonymous_profile_df = anonymous_profile_df.drop(columns=['gender', 'age', 'income']) # Drop columns with NaN values from anonymous_profile_df

# Remove rows with NaN values from profile_df
profile_df = profile_df.dropna()

In [ ]:
profile_df

In [ ]:
anonymous_profile_df

In [ ]:
transcript_df

In [ ]:
transcript_df = transcript_df.rename(columns={'person': 'customer_id'}) # Rename 'person' column to 'customer_id' for clarity
transaction_df = transcript_df[transcript_df['event'] == 'transaction'].copy() # Extract only transaction events into a separate dataframe
offer_transcript_df = transcript_df[transcript_df['event'] != 'transaction'].copy() # Extract only offer-related events into a separate dataframe

In [ ]:
# Extract the amount from the 'value' column 
transaction_df['amount'] = transaction_df['value'].apply(lambda x: x.get('amount', 0) if isinstance(x, dict) else 0)
transaction_df = transaction_df[['customer_id', 'amount', 'time']].reset_index(drop=True)

# Calculate total spending for each customer_id
customer_spending = transaction_df.groupby('customer_id')['amount'].sum().reset_index()
customer_spending.columns = ['customer_id', 'total_spending']

In [ ]:
transaction_df

In [ ]:
# Extract offer_id from the 'value' column (handle both 'offer id' and 'offer_id' keys)
offer_transcript_df['offer_id'] = offer_transcript_df['value'].apply(
    lambda x: x.get('offer_id') or x.get('offer id') if isinstance(x, dict) else None
)
offer_transcript_df = offer_transcript_df.drop(columns=['value'])
offer_transcript_df = offer_transcript_df[['customer_id','event','offer_id','time']].reset_index(drop=True)
offer_df = offer_transcript_df.merge(portfolio_df, on='offer_id', how='left')


In [ ]:
profile_df = profile_df.merge(customer_spending, on='customer_id', how='left')
anonymous_profile_df = anonymous_profile_df.merge(customer_spending, on='customer_id', how='left')

In [ ]:
profile_df 

In [ ]:
anonymous_profile_df

In [ ]:
portfolio_df

In [ ]:
transaction_df

In [ ]:
customer_spending

In [ ]:
# Check if every customer_id in profile_df and anonymous_profile_df has at least one transaction event

# Get all unique customer_ids from profile_df and anonymous_profile_df
profile_customer_ids = set(profile_df['customer_id'].unique())
anonymous_profile_customer_ids = set(anonymous_profile_df['customer_id'].unique())
all_profile_customers = profile_customer_ids.union(anonymous_profile_customer_ids)

# Get all customer_ids with transaction events
transaction_customer_ids = set(transaction_df['customer_id'].unique())

# Find customers without transaction events
missing_in_transaction_profile = profile_customer_ids - transaction_customer_ids
missing_in_transaction_anonymous = anonymous_profile_customer_ids - transaction_customer_ids
missing_in_transaction_all = all_profile_customers - transaction_customer_ids

print(f"Total customers in profile_df: {len(profile_customer_ids)}")
print(f"Total customers in anonymous_profile_df: {len(anonymous_profile_customer_ids)}")
print(f"Total profile customers (both): {len(all_profile_customers)}")
print(f"\nTotal customers with transaction events: {len(transaction_customer_ids)}")

print(f"\n=== CUSTOMERS WITHOUT TRANSACTION EVENTS ===")
print(f"Profile customers without transaction: {len(missing_in_transaction_profile)}")
if missing_in_transaction_profile:
    print(f"  Examples: {list(missing_in_transaction_profile)[:5]}")

print(f"\nAnonymous profile customers without transaction: {len(missing_in_transaction_anonymous)}")
if missing_in_transaction_anonymous:
    print(f"  Examples: {list(missing_in_transaction_anonymous)[:5]}")

print(f"\n=== SUMMARY ===")
if len(missing_in_transaction_all) == 0:
    print("✅ ALL profiles have at least one transaction event!")
else:
    print(f"❌ {len(missing_in_transaction_all)} profiles do NOT have any transaction event")
    print(f"   Percentage: {len(missing_in_transaction_all) / len(all_profile_customers) * 100:.2f}%")